# 3장 실습: 해석가능한 모델

『설명가능한 인공지능(XAI)』 3장 실습 노트북입니다. 실습 저장소: https://github.com/neomoon40/xai-textbook

로지스틱 회귀, 결정트리, EBM을 1장과 같은 대출 심사 데이터(German Credit)와 신청자 A로 학습하고, 다섯 모델을 비교합니다. 셀을 위에서부터 차례로 실행하세요. 뒤의 셀은 앞 셀이 만든 데이터와 모델을 씁니다. 모두 실행하는 데 Colab CPU 런타임에서 약 2~3분 걸립니다(EBM 학습 1~2분).

## 0. 라이브러리 설치와 버전 맞추기 (Colab)

이 책의 결과는 Python 3.13, scikit-learn 1.9.1, pandas 3.0.5, NumPy 2.5.3, interpret 0.7.8에서 얻었습니다. Colab에는 interpret가 없고 다른 라이브러리도 버전이 다르므로, **새 세션을 열 때마다** 아래 셀을 가장 먼저 실행합니다.

1. 아래 셀의 둘째 줄과 셋째 줄, **두 줄 모두** 맨 앞 `#`을 지우고 실행합니다.
2. 빨간 글씨로 `ERROR: pip's dependency resolver ...` 경고가 나올 수 있습니다. 마지막 줄에 `Successfully installed`가 나오면 설치는 끝난 것입니다.
3. **[런타임] → [세션 다시 시작]**을 누른 뒤, 이 셀은 건너뛰고 코드 3-1부터 실행합니다. 세션을 다시 시작해도 설치한 패키지는 남아 있습니다.

내 컴퓨터에서 부록 B의 requirements.txt로 이미 설치했다면 이 셀은 실행하지 않습니다.

In [ ]:
# Colab에서는 다음 두 줄의 #을 지우고 실행한다
# !pip install scikit-learn==1.9.1 pandas==3.0.5 numpy==2.5.3
# !pip install interpret==0.7.8 koreanize-matplotlib


### 실습 1. 준비와 로지스틱 회귀

#### 코드 3-1 준비: 패키지, 데이터 분할, 신청자 A

코드 3-1의 버전 출력이 scikit-learn 1.9.1, interpret 0.7.8이 아니면 설치 셀을 실행했는지, 세션을 다시 시작했는지 확인합니다.

In [ ]:
import numpy as np, pandas as pd, sklearn, interpret
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
print("scikit-learn", sklearn.__version__,
      "/ interpret", interpret.__version__)

df = fetch_openml("credit-g", version=1, as_frame=True).frame
X = df.drop(columns="class"); y = (df["class"] == "good").astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
A = 87                    # 신청자 A(1장 코드 1-5에서 찾은 인덱스)
x_a = X_test.loc[[A]]

def prep(scale=False, drop=None):     # 1장 코드 1-3과 같은 전처리
    cat = make_column_selector(dtype_include="category")
    num = make_column_selector(dtype_include="number")
    return make_column_transformer(
        (OneHotEncoder(drop=drop, handle_unknown="ignore"), cat),
        (StandardScaler() if scale else "passthrough", num),
        verbose_feature_names_out=False)
print(x_a[["checking_status", "duration", "credit_amount",
           "purpose", "age"]].to_string())


#### 코드 3-2 로지스틱 회귀의 오즈비

In [ ]:
lr = make_pipeline(prep(scale=True, drop="first"),
                   LogisticRegression(max_iter=1000))
lr.fit(X_train, y_train)
coef = pd.Series(lr[-1].coef_[0], index=lr[0].get_feature_names_out())
odds = np.exp(coef).sort_values()
acc = accuracy_score(y_test, lr.predict(X_test))
print(f"정확도 {acc:.3f}, 계수 {len(coef)}개")
print("오즈비 상위 5\n", odds.tail(5)[::-1].round(2).to_string())
print("오즈비 하위 5\n", odds.head(5).round(2).to_string())


### 실습 2. 결정트리의 깊이와 불안정성

#### 코드 3-3 트리의 깊이와 부트스트랩 트리 100개

In [ ]:
for k in range(1, 11):
    t = make_pipeline(prep(), DecisionTreeClassifier(
        max_depth=k, random_state=42)).fit(X_train, y_train)
    print(f"깊이 {k:2d}  노드 {t[-1].tree_.node_count:3d}  "
          f"학습 {t.score(X_train, y_train):.3f}  "
          f"평가 {t.score(X_test, y_test):.3f}")

rng = np.random.RandomState(42); roots = []; approve_a = 0
for b in range(100):            # 부트스트랩 표본 100개로 다시 학습
    i = rng.choice(len(X_train), len(X_train), replace=True)
    t = make_pipeline(prep(), DecisionTreeClassifier(
        max_depth=3, random_state=42))
    t.fit(X_train.iloc[i], y_train.iloc[i])
    roots.append(t[0].get_feature_names_out()[t[-1].tree_.feature[0]])
    approve_a += int(t.predict(x_a)[0])
print(pd.Series(roots).value_counts().to_string())
print("A를 승인한 트리:", approve_a, "/ 100")


### 실습 3. EBM의 형상 함수와 항별 기여

#### 코드 3-4 EBM 학습과 항 중요도

EBM 학습에 1~2분 걸립니다. `ebm`과 `ebm0`은 한 번 학습한 뒤 다시 학습하지 않습니다. 버전이 같아도 EBM의 점수는 CPU에 따라 끝자리가 조금 다를 수 있습니다(책의 숫자는 Colab CPU 런타임 기준).

In [ ]:
from interpret.glassbox import ExplainableBoostingClassifier
EBC = ExplainableBoostingClassifier
ebm = EBC(random_state=42).fit(X_train, y_train)
ebm0 = EBC(random_state=42, interactions=0).fit(X_train, y_train)
for name, m in [("EBM(기본 설정)", ebm), ("EBM(주효과만)", ebm0)]:
    p = m.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, m.predict(X_test))
    n = len(m.term_names_)
    print(f"{name:<12} 정확도 {acc:.3f}  "
          f"AUC {roc_auc_score(y_test, p):.3f}  항 {n}개")
imp = pd.Series(ebm.term_importances(), index=ebm.term_names_)
print(imp.sort_values(ascending=False).head(5).round(3).to_string())


#### 코드 3-5 대출 기간의 형상 함수

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib      # 그래프의 한글 표시(첫 셀에서 설치)
j = ebm.term_names_.index("duration")
cuts = ebm.bins_[j][0]                       # 구간 경계
# 구간 점수(맨 앞 결측, 맨 뒤 미지 구간 제외)와 표준편차
score = ebm.term_scores_[j][1:-1]
sd = ebm.standard_deviations_[j][1:-1]
edges = np.concatenate([[X_train.duration.min()], cuts,
                        [X_train.duration.max()]])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.stairs(score, edges, baseline=None, color="navy", linewidth=2)
ax.stairs(score + sd, edges, baseline=score - sd, fill=True,
          alpha=0.2, color="gray")
ax.axhline(0, color="black", linewidth=0.8)
a_score = score[np.searchsorted(cuts, 36, side="right")]
ax.scatter([36], [a_score], color="white", edgecolor="navy", zorder=3)
ax.set_xlabel("대출 기간(개월)"); ax.set_ylabel("로짓 점수")
plt.savefig("code3-5.png", dpi=120, bbox_inches="tight")
tab = pd.DataFrame({"구간 시작": edges[:-1], "점수": score.round(3)})
print(tab.iloc[19:24].to_string(index=False))


#### 코드 3-6 신청자 A의 항별 기여

In [ ]:
loc = ebm.explain_local(x_a, y_test.loc[[A]]).data(0)
contrib = pd.Series(loc["scores"], index=loc["names"])
b0 = ebm.intercept_[0]; logit = b0 + contrib.sum()
print(f"승인 확률 {ebm.predict_proba(x_a)[0, 1]:.2f}, 로짓 {logit:.3f}"
      f" = 절편 {b0:.3f} + 항별 기여 합 {contrib.sum():.3f}")
print(contrib.sort_values().head(5).round(3).to_string())


### 실습 4. 다섯 모델 비교

#### 코드 3-7 다섯 모델의 성능과 A의 승인 확률

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
rf = RandomForestClassifier(n_estimators=300, random_state=42)
models = {
    "결정트리(깊이 3)": make_pipeline(prep(), dt),
    "로지스틱 회귀": make_pipeline(prep(scale=True),
                                LogisticRegression(max_iter=1000)),
    "EBM(주효과만)": ebm0, "EBM(기본 설정)": ebm,
    "랜덤 포레스트(300그루)": make_pipeline(prep(), rf),
}
rows = []
for name, m in models.items():
    if not name.startswith("EBM"): m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    rows.append({"모델": name,
                 "정확도": accuracy_score(y_test, m.predict(X_test)),
                 "AUC": roc_auc_score(y_test, p),
                 "A의 승인 확률": m.predict_proba(x_a)[0, 1]})
print(pd.DataFrame(rows).round(3).to_string(index=False))


### 더 해 보기

1. `interactions=0`, `interactions=10`, 기본 설정의 EBM을 비교해 정확도, AUC, 항의 수, A의 승인 확률을 정리해 보세요.
2. 나이(`age`)와 대출 금액(`credit_amount`)의 형상 함수를 코드 3-5처럼 그리고, 직관에 어긋나는 구간에 학습 데이터가 몇 명 있는지 세어 보세요.
3. 실습 2의 깊이 3 트리에서 A가 거친 경로(`decision_path`)를 출력하고 코드 3-6의 항별 기여와 비교해 보세요.